<a href="https://colab.research.google.com/github/e23189uop/Statistical-Learning-e23189/blob/main/assignment%2306_e23189/assignment06.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Part 01: Gaussian Process Regression

In [20]:
import kagglehub

# Download latest version
kagglepath="elikplim/eergy-efficiency-dataset"
path = kagglehub.dataset_download(kagglepath)

print("Path to dataset files:", path)

import os
print(f"Listing contents of: {path}")
!ls {path}
df2=pd.read_csv(path+"/ENB2012_data.csv")

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern, ConstantKernel as C, WhiteKernel
from sklearn.metrics import mean_squared_error, r2_score

# 1. Map columns safely
# Features: X1 to X8 | Targets: Y1 (Heating), Y2 (Cooling)
X = df2[['X1', 'X2', 'X3', 'X4', 'X5', 'X6', 'X7', 'X8']]
y_heating = df2['Y1']
y_cooling = df2['Y2']

# 2. Train/Test Splits
X_train, X_test, y_train_h, y_test_h = train_test_split(X, y_heating, test_size=0.2, random_state=42)
_, _, y_train_c, y_test_c = train_test_split(X, y_cooling, test_size=0.2, random_state=42)

# 3. Feature Scaling (Crucial for distance-based GP kernels)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 4. Construct Kernel (Matérn 3/2 captures non-linear trends; WhiteKernel absorbs noise)
kernel = C(1.0, (1e-2, 1e2)) * Matern(length_scale=1.0, nu=1.5) + WhiteKernel(noise_level=1e-1)

# 5. Initialize and fit separate Gaussian Process Models
gpr_heating = GaussianProcessRegressor(kernel=kernel, n_restarts_optimizer=10, random_state=42)
gpr_cooling = GaussianProcessRegressor(kernel=kernel, n_restarts_optimizer=10, random_state=42)

print("Fitting GPR for Heating Load...")
gpr_heating.fit(X_train_scaled, y_train_h)

print("Fitting GPR for Cooling Load...")
gpr_cooling.fit(X_train_scaled, y_train_c)

# 6. Evaluation
pred_h, std_h = gpr_heating.predict(X_test_scaled, return_std=True)
pred_c, std_c = gpr_cooling.predict(X_test_scaled, return_std=True)

print("\n================ GPR EVALUATION ================")
print(f"Heating Load (Y1) -> R² Score: {r2_score(y_test_h, pred_h):.4f} | MSE: {mean_squared_error(y_test_h, pred_h):.4f}")
print(f"Cooling Load (Y2) -> R² Score: {r2_score(y_test_c, pred_c):.4f} | MSE: {mean_squared_error(y_test_c, pred_c):.4f}")

Using Colab cache for faster access to the 'eergy-efficiency-dataset' dataset.
Path to dataset files: /kaggle/input/eergy-efficiency-dataset
Listing contents of: /kaggle/input/eergy-efficiency-dataset
ENB2012_data.csv
Fitting GPR for Heating Load...


/usr/local/lib/python3.12/dist-packages/sklearn/gaussian_process/kernels.py:452: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__k1__constant_value is close to the specified upper bound 100.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/gaussian_process/kernels.py:442: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__noise_level is close to the specified lower bound 1e-05. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(


Fitting GPR for Cooling Load...

================ GPR EVALUATION ================
Heating Load (Y1) -> R² Score: 0.9959 | MSE: 0.4250
Cooling Load (Y2) -> R² Score: 0.9812 | MSE: 1.7462


/usr/local/lib/python3.12/dist-packages/sklearn/gaussian_process/kernels.py:452: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__k1__constant_value is close to the specified upper bound 100.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/gaussian_process/kernels.py:442: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__noise_level is close to the specified lower bound 1e-05. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(


#Part 02: Linear Regression

In [18]:
import kagglehub

# Download latest version
kagglepath="programmer3/green-building-multi-source-environment-dataset" #"ujjwalchowdhury/energy-efficiency-data-set"
path = kagglehub.dataset_download(kagglepath)

print("Path to dataset files:", path)

import os
print(f"Listing contents of: {path}")
!ls {path}
df2=pd.read_csv(path+"/green_building_dataset.csv")

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

# 1. Screen numerical parameters
numeric_df = df2.select_dtypes(include=[np.number])

# 2. Automated Feature Evaluation via Pearson Correlation
target_col = 'predicted_energy_demand'
correlations = numeric_df.corr()[target_col].sort_values(ascending=False)

print("--- Feature Correlations with Target ---")
print(correlations)

# 3. Parameter Selection (Dropping target, and avoiding structural multi-collinearity)
X_lr = numeric_df.drop(columns=[target_col])
y_lr = numeric_df[target_col]

# 4. Data Partitioning
X_train_l, X_test_l, y_train_l, y_test_l = train_test_split(X_lr, y_lr, test_size=0.2, random_state=42)

# 5. Model Execution
lr_model = LinearRegression()
lr_model.fit(X_train_l, y_train_l)
y_pred_l = lr_model.predict(X_test_l)

# 6. Performance Summary
print("\n================ LINEAR REGRESSION RESULTS ================")
print(f"R² Score: {r2_score(y_test_l, y_pred_l):.4f}")
print(f"Mean Absolute Error (MAE): {mean_absolute_error(y_test_l, y_pred_l):.4f}")
print(f"Mean Squared Error (MSE): {mean_squared_error(y_test_l, y_pred_l):.4f}")

# 7. Coefficient Attribution
coef_df = pd.DataFrame({'Parameter': X_lr.columns, 'Weight/Coefficient': lr_model.coef_})
print("\n--- Model Coefficients ---")
print(coef_df.sort_values(by='Weight/Coefficient', ascending=False).to_string(index=False))

Using Colab cache for faster access to the 'green-building-multi-source-environment-dataset' dataset.
Path to dataset files: /kaggle/input/green-building-multi-source-environment-dataset
Listing contents of: /kaggle/input/green-building-multi-source-environment-dataset
green_building_dataset.csv
--- Feature Correlations with Target ---
predicted_energy_demand    1.000000
ventilation_rate           0.728865
electricity_consumption    0.398703
cooling_energy             0.370632
heating_energy             0.271304
equipment_load             0.058766
occupancy                  0.057655
activity_level             0.018522
wind_speed                 0.011333
indoor_humidity            0.007899
outdoor_temperature        0.006786
outdoor_humidity           0.006451
solar_radiation            0.005331
predicted_comfort_index    0.003568
rainfall                  -0.004161
indoor_temperature        -0.008106
indoor_lighting           -0.020631
indoor_noise              -0.024454
co2_concentrat